# Kombinle modeli eğitimi

Bu defter, Kombinle'nin telefonda internetsiz çalışan kıyafet tanıma modelini eğitir.

**Ne öğreniyor?**
- Kıyafetin türü: elbise, kot, etek, ceket, ayakkabı, çanta, gözlük, saat…
- Deseni: düz, çizgili, kareli, çiçekli, puantiyeli, hayvan deseni ya da başka desen

Renk modelde yok; uygulama rengi fotoğraftan kendisi hesaplıyor (daha tutarlı).

**Nasıl çalıştırılır?**
1. Üst menü → **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seç.
2. **Çalışma zamanı → Tümünü çalıştır**.
3. Yaklaşık 1–1,5 saat sürer. Sekmeyi açık bırak.
4. Son hücre bir GitHub anahtarı ister ve modeli Kombinle reposuna gönderir. GitHub yeni APK'yı kendisi derler.

**Veri:** [Fashionpedia](https://fashionpedia.github.io/home/). Etiketleri CC BY 4.0 lisanslı.
Fotoğrafların her birinin kendi lisansı var; ticari kullanıma kapalı olanlar eğitime alınmıyor.
Uygulamada atıf "Lisanslar" bölümünde yer alıyor.

In [ ]:
# ---------- Ayarlar ----------
import os, sys, json, re, random, time, glob, math, shutil, subprocess, zipfile
from collections import Counter, defaultdict

REPO = 'aliyuksekal/kombinle'        # modelin gönderileceği repo
DAL = 'main'
GIRDI = 224                          # modelin gördüğü resim boyutu (piksel)
KAYIT_BOYU = 256                     # kırpılmış örneklerin diske yazıldığı boyut
SINIF_BASI_EN_FAZLA = 4000           # her türden en çok kaç örnek (dengeli eğitim)
SINIF_BASI_EN_AZ = 150               # bundan az örneği olan tür modele alınmaz
EN_KUCUK_KUTU = 48                   # bundan küçük (piksel) kıyafet kutuları atlanır
DONEM_ISINMA = 3                     # 1. aşama: yalnız yeni katmanlar eğitilir
DONEM_INCE = 8                       # 2. aşama: modelin üst katmanları da ince ayarlanır
TOPLU = 64
TOHUM = 7

SINAMA = os.environ.get('KB_SINAMA') == '1'   # yalnız geliştirici sınaması: sahte küçük veri
CALISMA = '/content/kb' if os.path.isdir('/content') and not SINAMA else os.path.abspath(os.environ.get('KB_CALISMA', 'kb'))
VERI = os.path.join(CALISMA, 'fashionpedia')
KIRPIK = os.path.join(CALISMA, 'kirpik')
CIKTI = os.path.join(CALISMA, 'cikti')
for d in (VERI, KIRPIK, CIKTI):
    os.makedirs(d, exist_ok=True)
random.seed(TOHUM)
print('Çalışma klasörü:', CALISMA, '(sınama)' if SINAMA else '')

In [ ]:
# ---------- Ekran kartı kontrolü ----------
try:
    import tensorflow as tf
    gpu = tf.config.list_physical_devices('GPU')
    print('TensorFlow', tf.__version__, '| GPU:', gpu[0].name if gpu else 'YOK')
    if not gpu and not SINAMA:
        print('\n⚠️  GPU seçilmemiş. Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU.\n'
              '   GPU olmadan eğitim saatler sürer.')
except ImportError:
    tf = None
    print('TensorFlow yok (yalnız veri hazırlama sınanabilir).')

## 1. Veriyi indir
Eğitim fotoğrafları yaklaşık 3,5 GB. İndirme ve açma 10–15 dakika sürer.

In [ ]:
ADRESLER = {
    'train2020.zip': 'https://s3.amazonaws.com/ifashionist-dataset/images/train2020.zip',
    'val_test2020.zip': 'https://s3.amazonaws.com/ifashionist-dataset/images/val_test2020.zip',
    'instances_attributes_train2020.json': 'https://s3.amazonaws.com/ifashionist-dataset/annotations/instances_attributes_train2020.json',
    'instances_attributes_val2020.json': 'https://s3.amazonaws.com/ifashionist-dataset/annotations/instances_attributes_val2020.json',
}


def indir(ad, adres):
    hedef = os.path.join(VERI, ad)
    if os.path.exists(hedef) and os.path.getsize(hedef) > 0:
        print('var:', ad)
        return hedef
    print('indiriliyor:', ad)
    subprocess.run(['wget', '-q', '--show-progress', '-c', '-O', hedef + '.part', adres], check=True)
    os.replace(hedef + '.part', hedef)
    return hedef


def ac(zip_yolu, klasor):
    if os.path.isdir(klasor) and os.listdir(klasor):
        print('açık:', os.path.basename(klasor))
        return
    print('açılıyor:', os.path.basename(zip_yolu))
    with zipfile.ZipFile(zip_yolu) as z:
        z.extractall(VERI)


def sahte_veri_kur():
    """Geliştirici sınaması: Fashionpedia biçiminde küçük sahte veri üretir."""
    from PIL import Image, ImageDraw
    kat = ['shirt, blouse', 'top, t-shirt, sweatshirt', 'sweater', 'cardigan', 'jacket', 'vest', 'pants', 'shorts', 'skirt',
           'coat', 'dress', 'jumpsuit', 'cape', 'glasses', 'hat', 'headband, head covering, hair accessory', 'tie', 'glove',
           'watch', 'belt', 'leg warmer', 'tights, stockings', 'sock', 'shoe', 'bag, wallet', 'scarf', 'umbrella', 'hood', 'sleeve']
    oz = [('plain (pattern)', 'textile pattern'), ('stripe', 'textile pattern'), ('check', 'textile pattern'), ('floral', 'textile pattern'),
          ('dot', 'textile pattern'), ('leopard', 'textile pattern'), ('abstract', 'textile pattern'), ('jeans', 'nickname'),
          ('maxi (length)', 'length'), ('hoodie', 'nickname'), ('trench (coat)', 'nickname'), ('blazer', 'nickname'), ('leggings', 'nickname'),
          ('denim', 'textile material')]
    for bolum, klasor, n in (('train2020', 'train', 260), ('val2020', 'test', 60)):
        os.makedirs(os.path.join(VERI, klasor), exist_ok=True)
        resimler, notlar = [], []
        for i in range(n):
            ad = '%s_%05d.jpg' % (bolum, i)
            im = Image.new('RGB', (320, 400), (random.randint(150, 255),) * 3)
            dr = ImageDraw.Draw(im)
            for k in range(3):
                x, y = random.randint(0, 200), random.randint(0, 260)
                w, h = random.randint(30, 120), random.randint(30, 140)
                dr.rectangle([x, y, x + w, y + h], fill=tuple(random.randint(0, 255) for _ in range(3)))
                c = random.randrange(len(kat))
                a = random.sample(range(len(oz)), 2)
                notlar.append({'id': len(notlar) + 1, 'image_id': i + 1, 'category_id': c, 'attribute_ids': a,
                               'bbox': [x, y, w, h], 'area': w * h, 'iscrowd': 0, 'segmentation': []})
            im.save(os.path.join(VERI, klasor, ad), quality=85)
            resimler.append({'id': i + 1, 'file_name': ad, 'width': 320, 'height': 400, 'license': 1 + i % 3})
        js = {'info': {}, 'categories': [{'id': i, 'name': k, 'supercategory': 'x', 'level': 1} for i, k in enumerate(kat)],
              'attributes': [{'id': i, 'name': a, 'supercategory': s, 'level': 1} for i, (a, s) in enumerate(oz)],
              'licenses': [{'id': 1, 'name': 'Attribution License', 'url': 'http://creativecommons.org/licenses/by/2.0/'},
                           {'id': 2, 'name': 'Attribution-NonCommercial License', 'url': 'http://creativecommons.org/licenses/by-nc/2.0/'},
                           {'id': 3, 'name': 'No known copyright restrictions', 'url': 'http://flickr.com/commons/usage/'}],
              'images': resimler, 'annotations': notlar}
        json.dump(js, open(os.path.join(VERI, 'instances_attributes_%s.json' % bolum), 'w'))


if SINAMA:
    sahte_veri_kur()
else:
    for ad, adres in ADRESLER.items():
        indir(ad, adres)
    ac(os.path.join(VERI, 'train2020.zip'), os.path.join(VERI, 'train'))
    ac(os.path.join(VERI, 'val_test2020.zip'), os.path.join(VERI, 'test'))


def resim_klasoru(bolum):
    """Zip'ten çıkan klasör adını bulur (train / test)."""
    for aday in (['train', 'train2020'] if bolum == 'train2020' else ['test', 'val2020', 'val_test2020']):
        yol = os.path.join(VERI, aday)
        if os.path.isdir(yol):
            return yol
    raise FileNotFoundError('Resim klasörü bulunamadı: ' + bolum)


print({ad: os.path.getsize(os.path.join(VERI, ad)) // 1_000_000 for ad in os.listdir(VERI) if os.path.isfile(os.path.join(VERI, ad))}, 'MB')

## 2. Fashionpedia türlerini Kombinle türlerine eşle
Her modeldeki sınıf, uygulamadaki bir ya da birkaç türe karşılık gelir. Örneğin "gomlek_bluz" sınıfı
uygulamada önce *Bluz*, sonra *Gömlek* olarak önerilir. Ayakkabının alt türünü (spor, topuklu…)
Fashionpedia ayırmadığı için model yalnız "ayakkabı" der; alt türü uygulama kendi analiziyle seçer.

In [ ]:
# model sınıfı: (uygulamadaki tür adayları — ilk sıradaki öne çıkar, uygulamadaki grup)
SINIFLAR = {
    'tisort': (['tisort', 'bluz'], 'ust'),
    'sweat': (['sweat', 'kazak'], 'ust'),
    'askili': (['askili', 'bluz'], 'ust'),
    'gomlek_bluz': (['bluz', 'gomlek', 'tunik'], 'ust'),
    'kazak': (['kazak', 'sweat'], 'ust'),
    'hirka': (['hirka', 'kazak'], 'dis'),
    'blazer': (['blazer', 'ceket'], 'dis'),
    'ceket': (['ceket', 'blazer', 'mont'], 'dis'),
    'yelek': (['yelek'], 'dis'),
    'kaban': (['kaban', 'tren', 'mont'], 'dis'),
    'tren': (['tren', 'kaban'], 'dis'),
    'mont': (['mont', 'ceket'], 'dis'),
    'kot': (['kot'], 'alt'),
    'tayt': (['tayt', 'kumas'], 'alt'),
    'esofman': (['esofman'], 'alt'),
    'kumas': (['kumas', 'kot'], 'alt'),
    'sort': (['sort'], 'alt'),
    'etek': (['etek', 'uzunetek'], 'alt'),
    'uzunetek': (['uzunetek', 'etek'], 'alt'),
    'elbise': (['elbise', 'uzunelbise', 'tunik'], 'tek'),
    'uzunelbise': (['uzunelbise', 'elbise', 'ferace'], 'tek'),
    'tulum': (['tulum'], 'tek'),
    'ayakkabi': ([], 'ayk'),
    'canta': (['canta'], 'aks'),
    'atki': (['atki', 'esarp'], 'aks'),
    'bere': (['bere'], 'aks'),
    'gozluk': (['gozluk'], 'aks'),
    'eldiven': (['eldiven'], 'aks'),
    'saat': (['saat'], 'aks'),
    'kemer': (['kemer'], 'aks'),
    'kulotlu': (['kulotlu'], 'ic'),
    'corap': (['corap'], 'ic'),
}
DESENLER = ['duz', 'cizgili', 'kareli', 'cicekli', 'puantiyeli', 'hayvan', 'diger']
DESEN_KURAL = [
    ('cizgili', r'stripe|chevron|herringbone|pinstripe'),
    ('kareli', r'check|plaid|tartan|houndstooth|argyle|gingham|gingam'),
    ('cicekli', r'floral|flower|plant|paisley|toile'),
    ('puantiyeli', r'\bdot|polka'),
    ('hayvan', r'leopard|cheetah|zebra|giraffe|snake|python|croc|animal|tiger'),
]


def kombinle_sinifi(kat, oz):
    """Fashionpedia kategori adı + özellik adları → model sınıfı (ya da None)."""
    k = kat.lower()
    o = ' | '.join(sorted(oz))
    var = lambda desen: re.search(desen, o) is not None
    uzun = var(r'maxi|floor')
    if k.startswith('shirt'):
        return 'gomlek_bluz'
    if k.startswith('top'):
        if var(r'sweatshirt|hoodie'):
            return 'sweat'
        if var(r'tank|camisole|halter|tube \(top\)|sleeveless'):
            return 'askili'
        return 'tisort'
    if k.startswith('sweater'):
        return 'kazak'
    if k.startswith('cardigan'):
        return 'hirka'
    if k.startswith('jacket'):
        if var(r'blazer'):
            return 'blazer'
        if var(r'puffer|parka|down \(|quilted'):
            return 'mont'
        return 'ceket'
    if k.startswith('vest'):
        return 'yelek'
    if k.startswith('pants'):
        if var(r'jeans|denim'):
            return 'kot'
        if var(r'legging'):
            return 'tayt'
        if var(r'track|jogger|sweatpant'):
            return 'esofman'
        return 'kumas'
    if k.startswith('shorts'):
        return 'sort'
    if k.startswith('skirt'):
        return 'uzunetek' if uzun else 'etek'
    if k.startswith('coat'):
        if var(r'trench'):
            return 'tren'
        if var(r'puffer|parka|down \('):
            return 'mont'
        return 'kaban'
    if k.startswith('dress'):
        return 'uzunelbise' if uzun else 'elbise'
    if k.startswith('jumpsuit'):
        return 'tulum'
    tablo = {'glasses': 'gozluk', 'hat': 'bere', 'glove': 'eldiven', 'watch': 'saat', 'belt': 'kemer',
             'tights': 'kulotlu', 'sock': 'corap', 'shoe': 'ayakkabi', 'bag': 'canta', 'scarf': 'atki'}
    for bas, sinif in tablo.items():
        if k.startswith(bas):
            return sinif
    return None


def desen_sinifi(desen_ozleri):
    """Desen özellikleri → desen sınıfı; bilgi yoksa -1 (eğitimde sayılmaz)."""
    if not desen_ozleri:
        return -1
    o = ' | '.join(sorted(x.lower() for x in desen_ozleri))
    for ad, kural in DESEN_KURAL:
        if re.search(kural, o):
            return DESENLER.index(ad)
    if re.search(r'plain', o) and len(desen_ozleri) == 1:
        return DESENLER.index('duz')
    return DESENLER.index('diger')


def lisans_uygun(lisans):
    """Ticari kullanıma kapalı (NonCommercial) fotoğraflar eğitime alınmaz."""
    if not lisans:
        return False
    metin = (lisans.get('name', '') + ' ' + lisans.get('url', '')).lower()
    return not re.search(r'noncommercial|non-commercial|by-nc|/nc', metin)


def ornekleri_cikar(bolum, sinir=None):
    js = json.load(open(os.path.join(VERI, 'instances_attributes_%s.json' % bolum)))
    kat = {c['id']: c['name'] for c in js['categories']}
    oz_ad = {a['id']: a['name'].lower() for a in js['attributes']}
    oz_ust = {a['id']: (a.get('supercategory') or '').lower() for a in js['attributes']}
    lis = {l['id']: l for l in js.get('licenses', [])}
    resim = {r['id']: r for r in js['images']}
    if not hasattr(ornekleri_cikar, 'yazildi'):
        ornekleri_cikar.yazildi = True
        print('Desen özellikleri:', sorted(oz_ad[i] for i in oz_ad if 'pattern' in oz_ust[i])[:40])
        print('Lisanslar:', [(l['id'], l.get('name')) for l in js.get('licenses', [])])
    lisans_say, ornekler = Counter(), []
    for n in js['annotations']:
        r = resim.get(n['image_id'])
        if not r:
            continue
        uygun = lisans_uygun(lis.get(r.get('license')))
        lisans_say['uygun' if uygun else 'ticari değil / bilinmiyor'] += 1
        if not uygun:
            continue
        x, y, w, h = n['bbox']
        if w < EN_KUCUK_KUTU or h < EN_KUCUK_KUTU:
            continue
        ozler = [oz_ad[a] for a in n.get('attribute_ids', []) if a in oz_ad]
        sinif = kombinle_sinifi(kat.get(n['category_id'], ''), ozler)
        if not sinif:
            continue
        desen = desen_sinifi([oz_ad[a] for a in n.get('attribute_ids', []) if a in oz_ad and 'pattern' in oz_ust[a]])
        ornekler.append({'resim': r['file_name'], 'kutu': [x, y, w, h], 'sinif': sinif, 'desen': desen})
    print(bolum, 'lisans:', dict(lisans_say))
    if sinir:
        gruplu = defaultdict(list)
        for o in ornekler:
            gruplu[o['sinif']].append(o)
        ornekler = []
        for s, l in gruplu.items():
            random.shuffle(l)
            ornekler += l[:sinir]
    return ornekler


egitim_ornek = ornekleri_cikar('train2020', SINIF_BASI_EN_FAZLA)
dogrulama_ornek = ornekleri_cikar('val2020')
sayim = Counter(o['sinif'] for o in egitim_ornek)
SINIF_LISTESI = [s for s in SINIFLAR if sayim[s] >= (SINIF_BASI_EN_AZ if not SINAMA else 5)]
dusen = [s for s in SINIFLAR if s not in SINIF_LISTESI]
egitim_ornek = [o for o in egitim_ornek if o['sinif'] in SINIF_LISTESI]
dogrulama_ornek = [o for o in dogrulama_ornek if o['sinif'] in SINIF_LISTESI]
assert len(SINIF_LISTESI) != len(DESENLER), 'tür ve desen çıktıları aynı boyda olmamalı'
print('\nModele giren türler (%d):' % len(SINIF_LISTESI))
for s in SINIF_LISTESI:
    print('  %-12s %5d' % (s, sayim[s]))
print('Yeterli örneği olmadığı için çıkan:', dusen or 'yok')
print('Desen dağılımı:', Counter(DESENLER[o['desen']] if o['desen'] >= 0 else 'bilinmiyor' for o in egitim_ornek))
print('Eğitim:', len(egitim_ornek), '| Doğrulama:', len(dogrulama_ornek))

## 3. Kıyafetleri kırp
Her kıyafet kutusu biraz boşlukla kesilir ve beyaz zeminle kareye tamamlanır.
Uygulama da telefonda fotoğrafı aynı şekilde kareye tamamlıyor.

In [ ]:
from PIL import Image
import multiprocessing
from concurrent.futures import ProcessPoolExecutor


def kareye_tamamla(im, boy):
    im = im.convert('RGB')
    w, h = im.size
    o = boy / max(w, h)
    yeni = im.resize((max(1, round(w * o)), max(1, round(h * o))), Image.BILINEAR)
    zemin = Image.new('RGB', (boy, boy), (255, 255, 255))
    zemin.paste(yeni, ((boy - yeni.width) // 2, (boy - yeni.height) // 2))
    return zemin


def resmi_isle(is_):
    yol, gorevler = is_
    cikan = []
    try:
        with Image.open(yol) as im:
            im.load()
            W, H = im.size
            for hedef, (x, y, w, h) in gorevler:
                p = 0.08 * max(w, h)
                kutu = (max(0, x - p), max(0, y - p), min(W, x + w + p), min(H, y + h + p))
                kareye_tamamla(im.crop(tuple(int(round(v)) for v in kutu)), KAYIT_BOYU).save(hedef, quality=90)
                cikan.append(hedef)
    except Exception as e:
        print('atlandı:', os.path.basename(yol), e)
    return cikan


def kirp(ornekler, bolum):
    klasor = resim_klasoru(bolum)
    hedef_klasor = os.path.join(KIRPIK, bolum)
    os.makedirs(hedef_klasor, exist_ok=True)
    isler = defaultdict(list)
    for i, o in enumerate(ornekler):
        o['dosya'] = os.path.join(hedef_klasor, '%06d_%s.jpg' % (i, o['sinif']))
        if not os.path.exists(o['dosya']):
            isler[os.path.join(klasor, o['resim'])].append((o['dosya'], o['kutu']))
    bas = time.time()
    with ProcessPoolExecutor(max_workers=max(2, os.cpu_count() or 2), mp_context=multiprocessing.get_context('fork')) as h:
        for k, _ in enumerate(h.map(resmi_isle, list(isler.items()), chunksize=32)):
            if k and k % 2000 == 0:
                print('  %d / %d fotoğraf (%.0f sn)' % (k, len(isler), time.time() - bas))
    tamam = [o for o in ornekler if os.path.exists(o['dosya'])]
    print(bolum, ':', len(tamam), 'örnek hazır')
    return tamam


egitim_ornek = kirp(egitim_ornek, 'train2020')
dogrulama_ornek = kirp(dogrulama_ornek, 'val2020')

## 4. Modeli eğit
Temel: ImageNet'te eğitilmiş **MobileNetV3-Large** (telefonda hızlı). Üstüne iki çıkış eklenir:
tür ve desen. Önce yalnız yeni katmanlar, sonra modelin üst kısmı ince ayarlanır.

In [ ]:
if tf is None:
    raise SystemExit('TensorFlow gerekli (Colab\'da hazır gelir).')
import numpy as np
import keras
from keras import layers, ops

keras.utils.set_random_seed(TOHUM)
SINIF_NO = {s: i for i, s in enumerate(SINIF_LISTESI)}


def veri_kumesi(ornekler, egitim):
    yollar = [o['dosya'] for o in ornekler]
    tur = np.array([SINIF_NO[o['sinif']] for o in ornekler], dtype=np.int32)
    desen = np.array([o['desen'] for o in ornekler], dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((yollar, tur, desen))
    if egitim:
        ds = ds.shuffle(len(yollar), seed=TOHUM, reshuffle_each_iteration=True)

    def yukle(yol, t, d):
        im = tf.io.decode_jpeg(tf.io.read_file(yol), channels=3)
        im = tf.image.convert_image_dtype(im, tf.float32)       # 0..1
        if egitim:
            olcek = tf.random.uniform([], 0.85, 1.0)
            kenar = tf.cast(tf.cast(KAYIT_BOYU, tf.float32) * olcek, tf.int32)
            im = tf.image.random_crop(im, [kenar, kenar, 3])
            im = tf.image.random_flip_left_right(im)
            im = tf.image.random_brightness(im, 0.12)
            im = tf.image.random_contrast(im, 0.85, 1.15)
            im = tf.clip_by_value(im, 0.0, 1.0)
        im = tf.image.resize(im, [GIRDI, GIRDI], antialias=True) * 255.0   # model 0..255 bekler
        return im, {'tur': t, 'desen': d}

    return ds.map(yukle, num_parallel_calls=tf.data.AUTOTUNE).batch(TOPLU).prefetch(tf.data.AUTOTUNE)


def maskeli_desen_kaybi(y_dogru, y_tahmin):
    """Deseni bilinmeyen örnekler (-1) desen kaybına katılmaz."""
    y = ops.reshape(ops.cast(y_dogru, 'int32'), (-1,))
    gecerli = ops.cast(y >= 0, 'float32')
    k = keras.losses.sparse_categorical_crossentropy(ops.maximum(y, 0), y_tahmin)
    return ops.sum(k * gecerli) / ops.maximum(ops.sum(gecerli), 1.0)


def model_kur():
    taban = keras.applications.MobileNetV3Large(input_shape=(GIRDI, GIRDI, 3), include_top=False, weights='imagenet',
                                                 include_preprocessing=True, pooling='avg')
    taban.trainable = False
    giris = keras.Input((GIRDI, GIRDI, 3), name='resim')
    x = taban(giris, training=False)
    x = layers.Dropout(0.3)(x)
    tur = layers.Dense(len(SINIF_LISTESI), activation='softmax', name='tur')(x)
    desen = layers.Dense(len(DESENLER), activation='softmax', name='desen')(layers.Dropout(0.2)(x))
    return keras.Model(giris, {'tur': tur, 'desen': desen}), taban


egitim_ds = veri_kumesi(egitim_ornek, True)
dogrulama_ds = veri_kumesi(dogrulama_ornek, False)
model, taban = model_kur()


def derle(oran):
    model.compile(optimizer=keras.optimizers.Adam(oran),
                  loss={'tur': 'sparse_categorical_crossentropy', 'desen': maskeli_desen_kaybi},
                  loss_weights={'tur': 1.0, 'desen': 0.5},
                  metrics={'tur': ['accuracy']})


print('1. aşama: yeni katmanlar')
derle(1e-3)
model.fit(egitim_ds, validation_data=dogrulama_ds, epochs=1 if SINAMA else DONEM_ISINMA, verbose=1 if not SINAMA else 2)

print('2. aşama: ince ayar')
taban.trainable = True
for katman in taban.layers[:-60]:
    katman.trainable = False
for katman in taban.layers:
    if isinstance(katman, layers.BatchNormalization):
        katman.trainable = False
derle(1e-4)
model.fit(egitim_ds, validation_data=dogrulama_ds, epochs=1 if SINAMA else DONEM_INCE, verbose=1 if not SINAMA else 2,
          callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)])

## 5. Ne kadar doğru?
Eğitimde hiç görmediği doğrulama fotoğraflarıyla ölçülür.

In [ ]:
tahmin = model.predict(dogrulama_ds, verbose=0)
dogru_tur = np.array([SINIF_NO[o['sinif']] for o in dogrulama_ornek])
dogru_desen = np.array([o['desen'] for o in dogrulama_ornek])
tur_t = tahmin['tur'].argmax(1)
tur_dogruluk = float((tur_t == dogru_tur).mean())
m = dogru_desen >= 0
desen_dogruluk = float((tahmin['desen'].argmax(1)[m] == dogru_desen[m]).mean()) if m.any() else 0.0
duz = DESENLER.index('duz')
duz_desenli = float(((tahmin['desen'].argmax(1)[m] == duz) == (dogru_desen[m] == duz)).mean()) if m.any() else 0.0
emin = tahmin['tur'].max(1) >= 0.75
emin_dogruluk = float((tur_t[emin] == dogru_tur[emin]).mean()) if emin.any() else 0.0
print('Tür doğruluğu        : %%%.1f' % (tur_dogruluk * 100))
print('Desen türü doğruluğu : %%%.1f' % (desen_dogruluk * 100))
print('Düz / desenli ayrımı : %%%.1f' % (duz_desenli * 100))
print('Model emin (≥%%75) olduğunda: %%%.1f doğru, örneklerin %%%.0f\'inde emin' % (emin_dogruluk * 100, emin.mean() * 100))
print('\nTür bazında:')
for s, i in SINIF_NO.items():
    sec = dogru_tur == i
    if sec.any():
        print('  %-12s %%%5.1f  (%d örnek)' % (s, (tur_t[sec] == i).mean() * 100, sec.sum()))

## 6. Telefon modeline dönüştür
Model küçültülür (8 bit ağırlık) ve telefondaki sonuçla buradaki sonuç karşılaştırılır.

In [ ]:
kayit_yolu = os.path.join(CALISMA, 'kayitli_model')
shutil.rmtree(kayit_yolu, ignore_errors=True)
model.export(kayit_yolu)
donustur = tf.lite.TFLiteConverter.from_saved_model(kayit_yolu)
donustur.optimizations = [tf.lite.Optimize.DEFAULT]
tflite = donustur.convert()
MODEL_YOLU = os.path.join(CIKTI, 'kombinle_model.tflite')
open(MODEL_YOLU, 'wb').write(tflite)
print('Model boyutu: %.1f MB' % (len(tflite) / 1e6))

# telefondaki gibi çalıştırıp karşılaştır: çıkışlar boyutlarına göre ayırt edilir
yorumlayici = tf.lite.Interpreter(model_content=tflite)
yorumlayici.allocate_tensors()
giris_d = yorumlayici.get_input_details()[0]
cikis_d = yorumlayici.get_output_details()
assert list(giris_d['shape']) == [1, GIRDI, GIRDI, 3] and giris_d['dtype'] == np.float32, giris_d
boy = {int(c['shape'][-1]): c['index'] for c in cikis_d}
assert len(SINIF_LISTESI) in boy and len(DESENLER) in boy, boy
uyan = 0
ornek_say = min(200, len(dogrulama_ornek))
toplu = next(iter(veri_kumesi(dogrulama_ornek[:ornek_say], False).unbatch().batch(ornek_say)))[0].numpy()
keras_t = model.predict(toplu, verbose=0)['tur'].argmax(1)
for i in range(ornek_say):
    yorumlayici.set_tensor(giris_d['index'], toplu[i:i + 1])
    yorumlayici.invoke()
    uyan += int(yorumlayici.get_tensor(boy[len(SINIF_LISTESI)])[0].argmax() == keras_t[i])
print('Telefon modeli ile eğitim modeli aynı türü buldu: %d / %d' % (uyan, ornek_say))
assert uyan >= ornek_say * 0.9, 'Dönüştürmede sorun var; modeli göndermeyin.'

bilgi = {
    'surum': int(time.time()),
    'girdi': GIRDI,
    'olcek': '0-255',
    'zemin': '#ffffff',
    'siniflar': [{'id': s, 'turlar': SINIFLAR[s][0], 'grup': SINIFLAR[s][1]} for s in SINIF_LISTESI],
    'desenler': DESENLER,
    'dogruluk': {'tur': round(tur_dogruluk, 3), 'desen': round(desen_dogruluk, 3), 'duz_desenli': round(duz_desenli, 3),
                 'emin75': round(emin_dogruluk, 3), 'ornek': len(dogrulama_ornek)},
    'kaynak': 'Fashionpedia (etiketler CC BY 4.0)',
}
BILGI_YOLU = os.path.join(CIKTI, 'kombinle_model.json')
json.dump(bilgi, open(BILGI_YOLU, 'w'), ensure_ascii=False, indent=1)
print(json.dumps(bilgi['dogruluk']))

## 7. Modeli Kombinle'ye gönder
Bir **GitHub anahtarı** gerekiyor (bir kez):
1. GitHub → sağ üst fotoğraf → **Settings → Developer settings → Personal access tokens → Fine-grained tokens → Generate new token**
2. *Repository access*: **Only select repositories → kombinle**
3. *Permissions → Repository permissions → Contents*: **Read and write**
4. Oluşan anahtarı kopyala, aşağıdaki kutuya yapıştır (ekranda görünmez, hiçbir yere kaydedilmez).

İki dosya tek bir kayıtla `app/src/main/assets/` klasörüne gider; GitHub yeni APK'yı derler (4–5 dakika).
Anahtarla uğraşmak istemezsen alttaki hücrede iki dosyayı indirip GitHub'da elle de yükleyebilirsin.

In [ ]:
import base64, getpass, urllib.request


def github(yontem, yol, anahtar, govde=None):
    istek = urllib.request.Request('https://api.github.com/repos/%s%s' % (REPO, yol), method=yontem,
                                   data=json.dumps(govde).encode() if govde is not None else None,
                                   headers={'Authorization': 'Bearer ' + anahtar, 'Accept': 'application/vnd.github+json',
                                            'X-GitHub-Api-Version': '2022-11-28', 'User-Agent': 'kombinle-egitim'})
    with urllib.request.urlopen(istek, timeout=120) as y:
        return json.loads(y.read() or b'{}')


def gonder():
    anahtar = getpass.getpass('GitHub anahtarı: ').strip()
    if not anahtar:
        print('Anahtar girilmedi; gönderilmedi.')
        return
    son = github('GET', '/git/ref/heads/' + DAL, anahtar)['object']['sha']
    agac = github('GET', '/git/commits/' + son, anahtar)['tree']['sha']
    dosyalar = []
    for yerel, uzak in ((MODEL_YOLU, 'app/src/main/assets/kombinle_model.tflite'), (BILGI_YOLU, 'app/src/main/assets/kombinle_model.json')):
        blob = github('POST', '/git/blobs', anahtar, {'content': base64.b64encode(open(yerel, 'rb').read()).decode(), 'encoding': 'base64'})
        dosyalar.append({'path': uzak, 'mode': '100644', 'type': 'blob', 'sha': blob['sha']})
    yeni_agac = github('POST', '/git/trees', anahtar, {'base_tree': agac, 'tree': dosyalar})['sha']
    mesaj = 'Kombinle modeli: tür %%%.0f, desen %%%.0f doğruluk' % (bilgi['dogruluk']['tur'] * 100, bilgi['dogruluk']['desen'] * 100)
    kayit = github('POST', '/git/commits', anahtar, {'message': mesaj, 'tree': yeni_agac, 'parents': [son]})['sha']
    github('PATCH', '/git/refs/heads/' + DAL, anahtar, {'sha': kayit})
    print('Gönderildi ✓ GitHub yeni APK\'yı derliyor: https://github.com/%s/actions' % REPO)


if SINAMA:
    print('sınama: gönderim atlandı')
else:
    gonder()

In [ ]:
# İstersen dosyaları bilgisayarına indir (elle yüklemek için: GitHub → kombinle → app/src/main/assets → Add file → Upload files)
if not SINAMA:
    try:
        from google.colab import files
        files.download(MODEL_YOLU)
        files.download(BILGI_YOLU)
    except Exception as e:
        print('İndirme yalnız Colab\'da çalışır:', e)